# Training

How an outfit becomes a training example, what the model is, and what training produces.

**Prerequisites:** the three preprocessing scripts have run (see the README), so the outfit table and the strip archives exist. A full training run takes hours on a CPU (about 22 minutes per 100 epochs on an Apple M5 Pro GPU); set `RUN_TRAINING = True` below for a short demo run.

In [ ]:
import os
import sys
from pathlib import Path

# Make `src` and `scripts` importable when the kernel starts in `notebooks/`.
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_processing.schema import PRODUCT_CLASSES
from src.utils import visualization as viz
from src.utils.setup_utilities import load_config

from scripts.training.train_vae import train_outfit_embeddings
from src.data_processing.image_processing import load_strips
from src.generation.embeddings import load_training_embeddings
from src.models.vae256 import Conv_VAE, load_trained_model

In [ ]:
# Optional path to an alternative config file; defaults to `config.yml`.
CONFIG_PATH = os.environ.get("FASHION_CONFIG")

config = load_config(CONFIG_PATH)
training = config["training"]
pd.Series({key: training[key] for key in ("batch_size", "epochs", "learning_rate", "seed")}, name="value").to_frame()

## Outfits as images

Each outfit is a row with one product ID per slot (`NaN` when the slot is empty). Every product image is resized to 128 × 96 and the ten slots are concatenated side by side, with white for empty slots. The model therefore sees one 128 × 960 RGB image per outfit, as a 3 × 128 × 960 tensor, and each class always occupies the same position.

In [ ]:
outfits = pd.read_csv(config["data"]["outfits_w_products_path"], dtype={slot: str for slot in PRODUCT_CLASSES})
print(f"{len(outfits)} training outfits")
outfits.head(3)

In [ ]:
sample_strips = load_strips(training["train_sample_path"])
sample_indices = training["train_sample_indices"][: len(sample_strips)]
print("strip batch:", sample_strips.shape)
viz.show_strips(sample_strips, titles=[f"outfit row {i}" for i in sample_indices]);

## Model

A convolutional encoder ends in 512 values, split into a 256-dimensional mean and log-variance. The decoder maps a 256-dimensional vector back to the image. The two large dense layers (61,440 → 512 and 256 → 61,440) hold most of the weights.

The training loss is the mean pixel binary cross-entropy only; there is no KL term, so this is a stochastic autoencoder rather than a full VAE. Exported embeddings are the posterior means.

In [ ]:
model = Conv_VAE()


def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())


pd.Series(
    {"encoder": count_parameters(model.encoder), "decoder": count_parameters(model.decoder)},
    name="trainable parameters",
).to_frame()

## Training run

With `RUN_TRAINING = False` the notebook reads the artifacts of an earlier run. A demo run writes to `models/demo/` and `figures/demo/`, so existing weights are not overwritten.

In [ ]:
RUN_TRAINING = False
DEMO_OVERRIDES = {
    "epochs": 3,
    "model_weights_path": "./models/demo/vae_256.pt",
    "model_data_path": "./models/demo/vae_data_256.pkl",
    "epoch_images_path": "./figures/demo",
}

if RUN_TRAINING:
    train_outfit_embeddings(CONFIG_PATH, {"training": DEMO_OVERRIDES})
    training = load_config(CONFIG_PATH, {"training": DEMO_OVERRIDES})["training"]
print("weights:", training["model_weights_path"])

## Loss

In [ ]:
exported = load_training_embeddings(training["model_data_path"])
history = exported["reconstruction_loss"]
print(f"{len(history)} epochs, final loss {history[-1]:.5f}")
viz.plot_loss_curve(history);

## Reconstructions

Originals against reconstructions from the posterior mean. These are training outfits, so they measure fit rather than generalization.

In [ ]:
trained_model = load_trained_model(training["model_weights_path"])
reconstructions = viz.reconstruct(trained_model, sample_strips)
strips = [strip for pair in zip(sample_strips, reconstructions) for strip in pair]
titles = [f"{kind} {index}" for index in sample_indices for kind in ("original", "reconstruction")]
viz.show_strips(strips, titles=titles);

## Latent space

`latent_space` holds one 256-dimensional posterior mean per training outfit, in the row order of the outfit table. The projection below uses the first two principal components and colors each outfit by its number of products. The generation notebook measures distances in this space.

In [ ]:
latent_space = exported["latent_space"]
assert len(latent_space) == len(outfits), "embeddings and outfit table are out of sync"
projection = viz.pca_2d(latent_space)
products_per_outfit = outfits[list(PRODUCT_CLASSES)].notna().sum(axis=1)

fig, ax = plt.subplots(figsize=(6, 5))
points = ax.scatter(projection[:, 0], projection[:, 1], c=products_per_outfit, cmap="viridis", s=8)
fig.colorbar(points, label="products in outfit")
ax.set(xlabel="PC 1", ylabel="PC 2", title=f"{len(latent_space)} training outfits");

Next: `outfit-generation.ipynb` uses these weights and embeddings to generate outfits.